# Anomaly & outlier detection

Anomaly detection is **unsupervised**: find the points that don't fit the rest,
with no labels telling you which are anomalous. We build three complementary
detectors by hand and — importantly — show where they *disagree*:

- **Statistical** (z-score / IQR) — per-feature, simple, but blind to how
  features relate to each other.
- **Mahalanobis distance** — multivariate; accounts for feature covariance, so it
  catches points that are in-range on every axis but violate the *correlation*.
- **Distance/density** (k-NN) — model-free; flags points far from their neighbours.

The [EDA chapter](../01b-eda/exploratory-data-analysis.ipynb) already flagged
univariate outliers with the IQR rule; this chapter goes multivariate.

In [ ]:
:dep ndarray = { version = "0.15" }
:dep plotters = { version = "0.3", default-features = false, features = ["evcxr", "all_series", "all_elements"] }
use ndarray::{Array1, Array2};

// A correlated 2-D cloud (x1 ~ x0) plus three injected anomalies.
let mut data: Vec<Vec<f64>> = (0..70).map(|i| {
    let t = (i as f64 - 35.0) * 0.08;
    let n1 = (((i * 37) % 11) as f64 - 5.0) * 0.15;
    let n2 = (((i * 53) % 11) as f64 - 5.0) * 0.15;
    vec![t + n1, t + n2]
}).collect();
data.push(vec![5.0, 5.0]);    // A: far away, but ON the x0~x1 diagonal
data.push(vec![2.5, -2.5]);   // B: in-range on each axis, but OFF the diagonal
data.push(vec![-2.0, 2.6]);   // C: also a correlation violator
let n = data.len();

// Gauss-Jordan inverse (reused from the multi-output regression chapter).
fn inverse(m: &Array2<f64>) -> Array2<f64> {
    let k = m.nrows();
    let (mut a, mut inv) = (m.clone(), Array2::<f64>::eye(k));
    for col in 0..k {
        let piv = a[[col, col]];
        for j in 0..k { a[[col, j]] /= piv; inv[[col, j]] /= piv; }
        for row in 0..k { if row != col { let f = a[[row, col]]; for j in 0..k { a[[row, j]] -= f * a[[col, j]]; inv[[row, j]] -= f * inv[[col, j]]; } } }
    }
    inv
}
println!("{} points (2 features), 3 anomalies injected at the end", n);

## Statistical: the z-score rule

Flag any point more than 3 standard deviations from the mean **on any single
axis**. Cheap — but it has two blind spots we'll see immediately: the outliers
*themselves* inflate the standard deviation (the **masking effect**), and it
treats the features independently:

In [ ]:
{
    let mean = |j: usize| data.iter().map(|p| p[j]).sum::<f64>() / n as f64;
    let (m0, m1) = (mean(0), mean(1));
    let sd = |j: usize, m: f64| (data.iter().map(|p| (p[j] - m).powi(2)).sum::<f64>() / n as f64).sqrt();
    let (s0, s1) = (sd(0, m0), sd(1, m1));
    let flagged: Vec<usize> = (0..n).filter(|&i| ((data[i][0] - m0) / s0).abs() > 3.0 || ((data[i][1] - m1) / s1).abs() > 3.0).collect();
    println!("z-score flagged rows: {:?}", flagged);
    println!("-> it flagged NOTHING: the 3 injected outliers inflated the std enough to hide even");
    println!("   the far point A at (5,5) — the masking effect. And it's blind to the correlation");
    println!("   violators B, C at (2.5,-2.5)/(-2,2.6), whose coordinates look normal on each axis.");
}

## Mahalanobis distance (multivariate)

Mahalanobis distance measures how far a point is from the mean **in units of the
data's own covariance**: `d² = (x − μ)ᵀ Σ⁻¹ (x − μ)`. Because it uses `Σ⁻¹`, a
point off the correlation axis is *far* even if each coordinate looks normal —
exactly the case z-score missed. (For 2 features, `d² > 6` is roughly the 95%
cut-off.)

In [ ]:
{
    let (m0, m1) = (data.iter().map(|p| p[0]).sum::<f64>() / n as f64, data.iter().map(|p| p[1]).sum::<f64>() / n as f64);
    let c = |a: usize, b: usize| data.iter().map(|p| (p[a] - if a==0 {m0} else {m1}) * (p[b] - if b==0 {m0} else {m1})).sum::<f64>() / n as f64;
    let cov = Array2::from_shape_vec((2, 2), vec![c(0,0), c(0,1), c(0,1), c(1,1)]).unwrap();
    let inv = inverse(&cov);
    let d2 = |p: &[f64]| { let d = Array1::from(vec![p[0] - m0, p[1] - m1]); d.dot(&inv.dot(&d)) };
    let flagged: Vec<usize> = (0..n).filter(|&i| d2(&data[i]) > 6.0).collect();
    println!("Mahalanobis d^2 for the 3 injected points: A={:.1}, B={:.1}, C={:.1}", d2(&data[n-3]), d2(&data[n-2]), d2(&data[n-1]));
    println!("Mahalanobis flagged rows: {:?}", flagged);
    println!("-> catches ALL three, including the correlation violators z-score missed.");
}

## Distance-based: k-NN outlier score

A model-free alternative: score each point by its **average distance to its k
nearest neighbours**. Points in dense regions score low; isolated points score
high. No distribution assumption at all:

In [ ]:
{
    let dist = |a: &[f64], b: &[f64]| ((a[0]-b[0]).powi(2) + (a[1]-b[1]).powi(2)).sqrt();
    let k = 5usize;
    let score = |i: usize| {
        let mut ds: Vec<f64> = (0..n).filter(|&j| j != i).map(|j| dist(&data[i], &data[j])).collect();
        ds.sort_by(|a, b| a.partial_cmp(b).unwrap());
        ds[..k].iter().sum::<f64>() / k as f64
    };
    let mut scored: Vec<(usize, f64)> = (0..n).map(|i| (i, score(i))).collect();
    scored.sort_by(|a, b| b.1.partial_cmp(&a.1).unwrap());
    println!("top 5 k-NN outlier scores (row: mean dist to {} nearest):", k);
    for (i, s) in scored.iter().take(5) { println!("  row {:>2}: {:.2}", i, s); }
    println!("-> the injected anomalies (rows {}..{}) dominate the top scores.", n-3, n-1);
}

## Seeing the disagreement

Plot the cloud, marking the points Mahalanobis flags (red rings) — the ones
z-score flagged *none* of. The far point and the two correlation-violators all
stand out here, even though the violators sit *inside* the per-axis range but off
the diagonal:

In [ ]:
{
    use plotters::prelude::*;
    let (m0, m1) = (data.iter().map(|p| p[0]).sum::<f64>() / n as f64, data.iter().map(|p| p[1]).sum::<f64>() / n as f64);
    let c = |a: usize, b: usize| data.iter().map(|p| (p[a] - if a==0 {m0} else {m1}) * (p[b] - if b==0 {m0} else {m1})).sum::<f64>() / n as f64;
    let cov = Array2::from_shape_vec((2, 2), vec![c(0,0), c(0,1), c(0,1), c(1,1)]).unwrap();
    let inv = inverse(&cov);
    let d2 = |p: &[f64]| { let d = Array1::from(vec![p[0] - m0, p[1] - m1]); d.dot(&inv.dot(&d)) };
    evcxr_figure((460, 420), |root| {
        root.fill(&WHITE)?;
        let mut chart = ChartBuilder::on(&root).caption("Outliers: Mahalanobis-flagged in red", ("sans-serif", 14)).margin(10).x_label_area_size(30).y_label_area_size(40).build_cartesian_2d(-6f64..6f64, -6f64..6f64)?;
        chart.configure_mesh().x_desc("x0").y_desc("x1").draw()?;
        chart.draw_series((0..n).map(|i| {
            if d2(&data[i]) > 6.0 { Circle::new((data[i][0], data[i][1]), 6, RED.stroke_width(2)) }
            else { Circle::new((data[i][0], data[i][1]), 3, BLUE.mix(0.6).filled()) }
        }))?;
        Ok(())
    })
}

## Which detector, when

- **z-score / IQR** — fast first pass; univariate only. Fine when features are
  independent or you only care about extreme single values.
- **Mahalanobis** — when features are correlated and "normal on each axis but
  jointly weird" is a real failure mode. Assumes a roughly elliptical (Gaussian)
  cloud; struggles with multi-cluster or non-linear structure.
- **k-NN / density** — model-free, handles odd shapes, but scales as O(n²) naively
  and needs a distance that makes sense (scale your features first).

```{note}
For **time-series** anomalies (a spike in an otherwise smooth series), the
[`augurs`](../10-time-series/forecasting.ipynb) crate has a dedicated `outlier`
module — a maintained option where these general-purpose detectors would need
adapting to the temporal structure. Isolation Forest and LOF, common in Python,
have no maintained `smartcore`/`linfa` equivalent, so they're hand-rolled or
skipped here.
```